In [5]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from keras import Sequential
from keras.layers import InputLayer, Dense
from keras.utils import to_categorical
from keras.losses import categorical_crossentropy


In [ ]:
tf.keras.utils.set_random_seed(42)

dataset = pd.read_csv('fruit.csv')
print(dataset.head())

labels = dataset.pop('Class')
if 'Unnamed: 0' in dataset.columns:
    dataset = dataset.drop(columns=['Unnamed: 0'])

classes = sorted(labels.unique())
num_classes = len(classes)
labels_cat = keras.utils.to_categorical(labels, num_classes=num_classes)

train_dataset, test_dataset, train_labels, test_labels = train_test_split(
    dataset,
    labels_cat,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

train_dataset, validation_dataset, train_labels, validation_labels = train_test_split(
    train_dataset,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=np.argmax(train_labels, axis=1),
)

train_stats = train_dataset.describe()
train_stats = train_stats.transpose()
train_stats

def norm(x):
    return (x - train_stats['mean']) / train_stats['std']

normed_train_data = norm(train_dataset).astype('float32')
normed_validation_data = norm(validation_dataset).astype('float32')
normed_test_data = norm(test_dataset).astype('float32')

num_attributes = normed_train_data.shape[1]
learning_rates = [0.001, 0.01, 0.1]

def build_model(learning_rate):
    model = Sequential([
        InputLayer(input_shape=(num_attributes,)),
        Dense(num_attributes, activation='relu'),
        Dense(32, activation='relu'),
        Dense(10, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=categorical_crossentropy,
        metrics=['accuracy'],
    )
    return model

validation_results = []
for learning_rate in learning_rates:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    model = build_model(learning_rate)
    history = model.fit(
        normed_train_data,
        train_labels,
        epochs=30,
        validation_data=(normed_validation_data, validation_labels),
        verbose=0,
    )
    best_validation_accuracy = max(history.history['val_accuracy'])
    best_epoch = history.history['val_accuracy'].index(best_validation_accuracy) + 1
    validation_results.append({
        'learning_rate': learning_rate,
        'best_validation_accuracy': best_validation_accuracy,
        'best_epoch': best_epoch,
    })

results_df = pd.DataFrame(validation_results)
print(results_df)

best_result = sorted(
    validation_results,
    key=lambda result: (-result['best_validation_accuracy'], result['learning_rate']),
)[0]
best_learning_rate = best_result['learning_rate']
print(f'Najbolji learning_rate: {best_learning_rate}')
print(f"Najbolja validaciona tacnost: {best_result['best_validation_accuracy']:.4f}")

train_dataset_full = pd.concat([train_dataset, validation_dataset])
train_labels_full = np.concatenate([train_labels, validation_labels])
train_stats = train_dataset_full.describe()
train_stats = train_stats.transpose()

normed_train_data_full = norm(train_dataset_full).astype('float32')
normed_test_data = norm(test_dataset).astype('float32')

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
final_model = build_model(best_learning_rate)
final_history = final_model.fit(
    normed_train_data_full,
    train_labels_full,
    epochs=30,
    verbose=0,
)

test_loss, test_accuracy = final_model.evaluate(normed_test_data, test_labels, verbose=0)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
